[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/14_model_evaluation/02_roc_pr_and_thresholds.ipynb)

# Bài 02 — ROC, Precision–Recall và chọn ngưỡng

**🎯 Mục tiêu**
- Xây **ROC curve** from scratch; hiểu **ROC-AUC** = xác suất một positive ngẫu nhiên được xếp trên một negative ngẫu nhiên.
- **Precision–Recall curve** và average precision (PR-AUC); vì sao PR phù hợp hơn khi **mất cân bằng** nặng.
- Chọn ngưỡng: Youden's J, tối đa F1, recall tối thiểu, **tối thiểu chi phí**.
- Metric cho xác suất: log-loss, Brier score.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/14_model_evaluation

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn import metrics as skm
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

from ml_course.datasets import make_fraud_transactions
from ml_course.visualization import set_style

set_style()
fraud = make_fraud_transactions(n=30000, fraud_rate=0.02, seed=1)
features = ["amount_vnd", "hour", "distance_from_home_km", "is_foreign", "n_tx_last_24h", "card_age_days"]
X = fraud[features].astype(float).assign(amount_vnd=lambda d: np.log(d.amount_vnd))
y = fraud["is_fraud"].to_numpy()
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0, stratify=y)
scores = {"Logistic Regression": LogisticRegression(max_iter=2000).fit(X_tr, y_tr).predict_proba(X_te)[:, 1],
          "HistGradientBoosting": HistGradientBoostingClassifier(random_state=0).fit(X_tr, y_tr).predict_proba(X_te)[:, 1]}
print(f"tỉ lệ positive (test): {y_te.mean():.2%}")

## 1. ROC curve from scratch

Với mọi ngưỡng $t$: **TPR** (recall) $= \frac{TP}{P}$ và **FPR** $= \frac{FP}{N}$. Sắp xếp điểm số giảm dần; hạ ngưỡng dần qua từng điểm — mỗi positive làm TPR tăng $1/P$, mỗi negative làm FPR tăng $1/N$.

In [ ]:
def roc_curve_scratch(y_true, s):
    order = np.argsort(-s, kind="mergesort")
    y_sorted, s_sorted = y_true[order], s[order]
    distinct = np.r_[np.flatnonzero(np.diff(s_sorted)), len(s_sorted) - 1]   # chỉ lấy điểm cuối của mỗi nhóm điểm số bằng nhau
    tps = np.cumsum(y_sorted)[distinct]
    fps = (distinct + 1) - tps
    tpr = np.r_[0, tps / y_true.sum()]
    fpr = np.r_[0, fps / (len(y_true) - y_true.sum())]
    return fpr, tpr


fig, ax = plt.subplots(figsize=(6, 6))
for name, s in scores.items():
    fpr, tpr = roc_curve_scratch(y_te, s)
    auc = np.trapezoid(tpr, fpr)
    ax.plot(fpr, tpr, label=f"{name}: AUC = {auc:.4f}")
    print(f"{name}: AUC scratch = {auc:.6f} | sklearn = {skm.roc_auc_score(y_te, s):.6f}")
ax.plot([0, 1], [0, 1], "k--", label="đoán ngẫu nhiên (AUC = 0.5)")
ax.set_xlabel("FPR = FP / N")
ax.set_ylabel("TPR = recall")
ax.legend(fontsize=8)
ax.set_title("ROC curve")
plt.show()

## 2. AUC = xác suất xếp hạng đúng

$$\text{AUC} = P\big(s(\mathbf x^+) > s(\mathbf x^-)\big)$$

— chọn ngẫu nhiên một positive và một negative, xác suất model cho positive điểm cao hơn. Đây là thống kê **Mann–Whitney U** chuẩn hoá. Hệ quả: AUC chỉ phụ thuộc **thứ tự** điểm số (bất biến với mọi biến đổi đơn điệu) và **không** phụ thuộc ngưỡng.

In [ ]:
s = scores["Logistic Regression"]
pos, neg = s[y_te == 1], s[y_te == 0]
rng = np.random.default_rng(0)
pairs_pos, pairs_neg = rng.choice(pos, 200_000), rng.choice(neg, 200_000)
print(f"P(s+ > s−) ước lượng từ 200 000 cặp ngẫu nhiên = {np.mean(pairs_pos > pairs_neg):.4f}")
print(f"ROC-AUC                                       = {skm.roc_auc_score(y_te, s):.4f}")
print(f"AUC sau khi lấy s³ (biến đổi đơn điệu)         = {skm.roc_auc_score(y_te, s**3):.4f}")

## 3. Precision–Recall curve

Khi positive rất hiếm, FPR = FP/N có **mẫu số khổng lồ** → hàng trăm báo động nhầm vẫn chỉ là FPR nhỏ → ROC trông "đẹp". Precision = TP/(TP + FP) phản ánh trực tiếp gánh nặng báo động nhầm.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
for name, s in scores.items():
    precision, recall, _ = skm.precision_recall_curve(y_te, s)
    ax.plot(recall, precision, label=f"{name}: AP = {skm.average_precision_score(y_te, s):.3f}")
ax.axhline(y_te.mean(), color="k", ls="--", label=f"ngẫu nhiên: AP = tỉ lệ positive = {y_te.mean():.3f}")
ax.set_xlabel("recall")
ax.set_ylabel("precision")
ax.legend(fontsize=8)
ax.set_title("Precision–Recall curve")
plt.show()

**Average precision (AP)** ≈ diện tích dưới PR curve; baseline ngẫu nhiên = tỉ lệ positive (không phải 0.5 như ROC).

## 4. ROC-AUC vs PR-AUC khi tỉ lệ positive thay đổi

Cùng **một** model (điểm số phân phối giống hệt cho mỗi lớp), chỉ thay đổi tỉ lệ positive trong tập đánh giá:

In [ ]:
rows = []
for rate in [0.5, 0.1, 0.02, 0.005]:
    n_neg = 20000
    n_pos = int(n_neg * rate / (1 - rate))
    sp = rng.choice(pos, n_pos, replace=True)
    sn = rng.choice(neg, n_neg, replace=True)
    yy = np.r_[np.ones(n_pos), np.zeros(n_neg)]
    ss = np.r_[sp, sn]
    rows.append({"tỉ lệ positive": rate, "ROC-AUC": skm.roc_auc_score(yy, ss), "PR-AUC (AP)": skm.average_precision_score(yy, ss)})
pd.DataFrame(rows).round(3)

ROC-AUC **không đổi** (chỉ phụ thuộc phân phối điểm của từng lớp), còn PR-AUC **giảm mạnh** khi positive hiếm — phản ánh đúng thực tế: cùng model, nhưng mỗi cảnh báo đúng phải trả giá bằng nhiều báo động nhầm hơn. → Mất cân bằng nặng: báo cáo **PR-AUC** (kèm tỉ lệ positive), không chỉ ROC-AUC.

## 5. Chọn ngưỡng

In [ ]:
s = scores["HistGradientBoosting"]
fpr, tpr, thr_roc = skm.roc_curve(y_te, s)
youden_t = thr_roc[np.argmax(tpr - fpr)]                                   # (a) Youden's J = TPR − FPR lớn nhất
prec, rec, thr_pr = skm.precision_recall_curve(y_te, s)
f1 = 2 * prec[:-1] * rec[:-1] / np.maximum(prec[:-1] + rec[:-1], 1e-12)
f1_t = thr_pr[np.argmax(f1)]                                               # (b) F1 lớn nhất
recall_t = thr_pr[np.flatnonzero(rec[:-1] >= 0.8)[-1]]                     # (c) ngưỡng cao nhất vẫn đạt recall ≥ 80%
cost_fp, cost_fn = 50_000, 2_000_000                                       # (d) chi phí: kiểm tra 1 cảnh báo vs mất 1 vụ gian lận
grid = np.linspace(0.001, 0.99, 500)
costs = [cost_fp * np.sum((s >= t) & (y_te == 0)) + cost_fn * np.sum((s < t) & (y_te == 1)) for t in grid]
cost_t = grid[int(np.argmin(costs))]

rows = []
for name, t in [("mặc định 0.5", 0.5), ("Youden's J", youden_t), ("max F1", f1_t), ("recall ≥ 80%", recall_t), ("min chi phí", cost_t)]:
    p_ = (s >= t).astype(int)
    rows.append({"cách chọn": name, "ngưỡng": t, "precision": skm.precision_score(y_te, p_, zero_division=0),
                 "recall": skm.recall_score(y_te, p_), "số cảnh báo": int(p_.sum()),
                 "chi phí (triệu VND)": (cost_fp * np.sum((p_ == 1) & (y_te == 0)) + cost_fn * np.sum((p_ == 0) & (y_te == 1))) / 1e6})
pd.DataFrame(rows).round(3)

- **Youden's J** coi FP và FN như nhau trên thang FPR/TPR — thường cho rất nhiều cảnh báo khi mất cân bằng.
- **Max F1**: cân bằng precision/recall nhưng **bỏ qua** chi phí thật.
- **Ràng buộc nghiệp vụ** (recall ≥ 80%, hoặc "tối đa 100 cảnh báo/ngày") rất phổ biến trong thực tế.
- **Min chi phí**: tốt nhất khi ước lượng được chi phí — với xác suất calibrated, ngưỡng tối ưu lý thuyết là $\frac{C_{FP}}{C_{FP} + C_{FN}}$.

> ⚠️ Ngưỡng phải được chọn trên **validation**, không phải test (ở đây dùng test cho gọn — chương 15 làm đúng quy trình).

## 6. Metric cho xác suất: log-loss và Brier

AUC chỉ đánh giá **thứ tự**; khi xác suất được dùng trực tiếp (tính kỳ vọng thiệt hại, định giá rủi ro) cần metric đánh giá **giá trị** xác suất:
- **Log-loss** $-\frac1n\sum[y\log p + (1 - y)\log(1 - p)]$ — phạt nặng dự đoán tự tin mà sai.
- **Brier score** $\frac1n\sum(p - y)^2$ — MSE của xác suất.

In [ ]:
for name, s in scores.items():
    print(f"{name:21s} ROC-AUC = {skm.roc_auc_score(y_te, s):.4f} | log-loss = {skm.log_loss(y_te, s):.4f} | Brier = {skm.brier_score_loss(y_te, s):.4f}")
s_bad = scores["HistGradientBoosting"] ** 0.3                              # cùng thứ tự, xác suất bị "thổi phồng"
print(f"{'HistGB với p^0.3':21s} ROC-AUC = {skm.roc_auc_score(y_te, s_bad):.4f} | log-loss = {skm.log_loss(y_te, s_bad):.4f} | Brier = {skm.brier_score_loss(y_te, s_bad):.4f}")

Cùng ROC-AUC nhưng log-loss/Brier tệ hơn hẳn → xác suất **chưa calibrated**. Calibration là chủ đề của chương 37.

## 🧠 Kiểm tra nhanh

1. ROC-AUC = 0.85 nghĩa là gì theo ngôn ngữ xếp hạng?
2. Vì sao ROC-AUC có thể "đẹp" trong khi model vô dụng về precision khi positive rất hiếm?
3. Baseline của PR-AUC (model ngẫu nhiên) bằng bao nhiêu?
4. $C_{FP} = 1$, $C_{FN} = 19$, xác suất calibrated: ngưỡng tối ưu là bao nhiêu?

## 📌 Tóm tắt

- ROC: TPR vs FPR mọi ngưỡng; AUC = $P(s^+ > s^-)$, bất biến với biến đổi đơn điệu, không phụ thuộc tỉ lệ lớp.
- PR curve/AP phản ánh gánh nặng báo động nhầm; nên dùng khi positive hiếm.
- Ngưỡng chọn theo chi phí hoặc ràng buộc nghiệp vụ, trên validation.
- Log-loss/Brier đánh giá chất lượng xác suất — AUC thì không.

## 📚 Tài liệu tham khảo

- Fawcett (2006) — *An introduction to ROC analysis*
- Saito & Rehmsmeier (2015) — *The Precision-Recall Plot Is More Informative than the ROC Plot When Evaluating Binary Classifiers on Imbalanced Datasets*
- Hand (2009) — *Measuring classifier performance: a coherent alternative to the area under the ROC curve*